# Praktikum 7 Gen AI: Membangun Chatbot Panduan Akademik dengan RAG
### Penggabungan *Foundation Model* dengan Pengetahuan Eksternal (*Retrieval-Augmented Generation*)

---

## Bagian 1: Materi Singkat (Konsep Dasar)

### 1. Apa itu *Foundation Model* (FM)?
***Foundation Model*** adalah model kecerdasan buatan (AI) berukuran sangat besar yang dilatih menggunakan jutaan hingga miliaran teks dari internet. Contoh paling populer dari *Foundation Model* untuk bahasa adalah **Large Language Model (LLM)** seperti **Gemini, GPT, Qwen, dan LLaMA**.

Karena dilatih dengan data internet yang sangat luas, *Foundation Model* sangat pintar dalam memahami bahasa manusia, merangkum teks, dan menjawab pertanyaan umum.

### 2. Apa Masalahnya Jika *Foundation Model* Dipakai Langsung untuk Chatbot Kampus?
Jika kita langsung memakai LLM biasa untuk menjawab pertanyaan mahasiswa tentang aturan kampus, akan muncul **3 masalah utama**:
1. **Tidak Tahu Data Internal Kampus:** LLM tidak pernah membaca Buku Pedoman Akademik kampus kita karena dokumen tersebut bersifat internal/privat.
2. **Pengetahuan Tidak Selalu *Update*:** Jika aturan kampus berubah semester ini, LLM tidak otomatis mengetahuinya.
3. **Halusinasi (*Mengarang Jawaban*):** Saat ditanya aturan yang tidak diketahuinya, LLM sering kali menebak-nebak dan mengarang jawaban yang terdengar meyakinkan padahal salah.

---

### 3. Apa itu RAG (*Retrieval-Augmented Generation*)?
**RAG** adalah teknik menggabungkan kecerdasan bahasa dari **Foundation Model** dengan **dokumen/data eksternal** milik kita sendiri (misalnya Buku Panduan Akademik).

> **Analogi Sederhana:**
> * **LLM Biasa (Tanpa RAG)** ibarat mahasiswa yang mengikuti **ujian *Closed-Book***: hanya mengandalkan hafalan di kepala. Jika lupa atau tidak pernah belajar materinya, ia cenderung mengarang jawaban.
> * **Chatbot dengan RAG** ibarat mahasiswa yang mengikuti **ujian *Open-Book***: saat diberi pertanyaan, ia membuka Buku Panduan Akademik terlebih dahulu, mencari halaman yang relevan, lalu merangkai jawaban berdasarkan isi buku tersebut.

---

### 4. Bagaimana Cara Kerja RAG? (4 Langkah Sederhana)

```
[Dokumen Panduan] ---> (1) Dipecah per Pasal ---> (2) Diubah jadi Vektor Angka (Database)
                                                                  |
[Pertanyaan Mahasiswa] ----------------------------> (3) Cari Pasal Paling Mirip (Retrieval)
                                                                  |
                                                     (4) Foundation Model Merangkai Jawaban (Generation)
```

1. **Chunking (Pemecahan Dokumen):** Dokumen panduan yang panjang dipecah menjadi potongan-potongan kecil (misalnya per pasal/paragraf) agar mudah dicari.
2. **Embedding & Vector Database:** Komputer tidak memahami makna kata secara langsung. Model *Embedding* mengubah setiap potongan teks menjadi daftar angka (**vektor**) yang mewakili **makna kalimatnya**, lalu menyimpannya di basis data khusus bernama **FAISS**.
3. **Retrieval (Pencarian Semantik):** Saat mahasiswa bertanya, pertanyaan tersebut juga diubah menjadi vektor angka, lalu sistem mencari potongan dokumen yang **maknanya paling mirip** dengan pertanyaan tersebut.
4. **Generation (Penyusunan Jawaban):** Potongan pasal yang ditemukan diberikan kepada *Foundation Model* dengan perintah tegas: *"Jawab pertanyaan mahasiswa HANYA berdasarkan kutipan dokumen ini!"*

---

### 5. Kamus Istilah Penting (Versi Pemula)
* **Chunk:** Potongan kecil teks hasil pemecahan dokumen panjang.
* **Embedding:** Proses mengubah teks menjadi deretan angka agar komputer bisa mengukur kemiripan makna antar kalimat.
* **FAISS:** Pustaka (*library*) buatan Meta AI yang berfungsi sebagai mesin pencari vektor super cepat.
* **Grounding:** Teknik "mengunci" jawaban AI agar patuh pada sumber dokumen yang diberikan dan tidak mengarang bebas.

---
### Persiapan Sebelum Menjalankan Kode di Google Colab
* Klik menu **`Runtime` $\rightarrow$ `Change runtime type` $\rightarrow$ `T4 GPU` $\rightarrow$ `Save`** agar model AI berjalan cepat dan lancar dalam Bahasa Indonesia.
* Jika tidak memakai GPU (menggunakan CPU biasa), notebook ini tetap bisa berjalan karena sudah disiapkan mode cadangan (*fallback*) otomatis.

---
## Bagian 2: Praktikum Step-by-Step

### Langkah 1: Instalasi Pustaka (*Library*) yang Dibutuhkan
Pada sel pertama ini, kita menginstal alat-alat bantu yang dibutuhkan Python:
* **`sentence-transformers`**: Untuk mengubah teks Bahasa Indonesia menjadi vektor makna (*embedding*).
* **`faiss-cpu`**: Untuk menyimpan vektor dan mencari dokumen yang paling relevan.
* **`transformers` & `accelerate`**: Untuk menjalankan *Foundation Model* secara gratis di Google Colab.
* **`google-genai`**: Opsional, jika nanti ingin menghubungkan chatbot dengan Google Gemini API.

In [1]:
# ==========================================
# LANGKAH 1: INSTALASI LIBRARY
# ==========================================
!pip install -q sentence-transformers faiss-cpu transformers accelerate google-genai
print("Instalasi library berhasil! Siap lanjut ke Langkah 2.")

Instalasi library berhasil! Siap lanjut ke Langkah 2.


---
### Langkah 2: Menyiapkan Pengetahuan Eksternal & Memecah Dokumen (*Chunking*)

Di sini kita membuat simulasi **Buku Panduan Akademik Kampus** yang berisi aturan-aturan penting mahasiswa (Kalender Akademik, SKS, KRS, Cuti Kuliah, Syarat Skripsi, Kelulusan, dan Plagiarisme).

Setiap aturan dipisahkan oleh **satu baris kosong (`\n\n`)**. Dengan cara ini, kita bisa memecah dokumen menjadi potongan-potongan kecil (**chunks**) hanya dengan perintah `.split("\n\n")`.

In [2]:
# ==========================================
# LANGKAH 2: DATA PANDUAN AKADEMIK & CHUNKING
# ==========================================
dokumen_panduan = """
[BAB I - Kalender Akademik] Tahun akademik dibagi menjadi Semester Ganjil (September - Januari), Semester Genap (Februari - Juni), dan Semester Antara (Juli - Agustus) yang bersifat opsional dengan beban maksimal 9 SKS.

[BAB I - Registrasi KRS] Setiap mahasiswa wajib mengisi Kartu Rencana Studi (KRS) secara daring sebelum perkuliahan dimulai. Mahasiswa yang tidak mengisi KRS tanpa keterangan resmi dinyatakan berstatus tidak aktif (mangkir).

[BAB II - Batas Maksimal SKS] Jumlah maksimal SKS yang dapat diambil semester berikutnya ditentukan berdasarkan IPS semester lalu: IPS >= 3,00 maksimal 24 SKS; IPS 2,50 - 2,99 maksimal 20 SKS; IPS 2,00 - 2,49 maksimal 18 SKS; dan IPS < 2,00 maksimal 15 SKS.

[BAB II - Perbaikan KRS] Perubahan rencana studi (tambah atau hapus mata kuliah) hanya boleh dilakukan pada masa Perbaikan KRS maksimal 2 minggu setelah kuliah perdana berlangsung dengan persetujuan Dosen Pembimbing Akademik (DPA).

[BAB III - Syarat Cuti Akademik] Cuti akademik adalah penundaan kuliah sementara atas izin Rektor. Mahasiswa boleh mengajukan cuti setelah menempuh minimal 2 semester aktif berturut-turut.

[BAB III - Durasi Cuti & Masa Studi] Total cuti akademik selama kuliah S1 maksimal 2 semester dan tidak boleh diambil berturut-turut (kecuali sakit berat dengan surat dokter). Masa cuti tidak mengurangi jatah masa studi. Masa studi normal S1 adalah 8 semester dan batas maksimalnya 14 semester (lewat 14 semester dinyatakan Drop Out).

[BAB IV - Syarat Mengambil Skripsi] Mahasiswa S1 boleh mengambil Skripsi jika sudah lulus minimal 110 SKS, IPK minimal 2,75, tidak ada nilai E, nilai D maksimal 2 mata kuliah, serta lulus mata kuliah Metodologi Penelitian minimal nilai B.

[BAB IV - Syarat Kelulusan S1] Syarat lulus Program Sarjana (S1) adalah menyelesaikan minimal 144 SKS, IPK akhir minimal 2,50, lulus sidang skripsi, dan memiliki sertifikat TOEFL ITP minimal skor 450.

[BAB V - Aturan Plagiarisme] Seluruh laporan tugas akhir wajib lolos uji kemiripan (similarity check) kampus dengan batas maksimal kemiripan 20 persen. Mahasiswa yang terbukti melakukan plagiarisme atau kecurangan ujian dikenakan sanksi mulai dari nilai E, skorsing 1-2 semester, hingga diberhentikan.
"""

# Memecah dokumen menjadi potongan-potongan kecil (chunks) per paragraf
chunks = [paragraf.strip() for paragraf in dokumen_panduan.strip().split("\n\n") if paragraf.strip()]

print(f"Dokumen berhasil dipecah menjadi {len(chunks)} potongan aturan (chunks).\n")
print("Contoh 2 potongan pertama:")
for i in range(2):
    print(f"Chunk {i+1}: {chunks[i]}\n")

Dokumen berhasil dipecah menjadi 9 potongan aturan (chunks).

Contoh 2 potongan pertama:
Chunk 1: [BAB I - Kalender Akademik] Tahun akademik dibagi menjadi Semester Ganjil (September - Januari), Semester Genap (Februari - Juni), dan Semester Antara (Juli - Agustus) yang bersifat opsional dengan beban maksimal 9 SKS.

Chunk 2: [BAB I - Registrasi KRS] Setiap mahasiswa wajib mengisi Kartu Rencana Studi (KRS) secara daring sebelum perkuliahan dimulai. Mahasiswa yang tidak mengisi KRS tanpa keterangan resmi dinyatakan berstatus tidak aktif (mangkir).



---
### Langkah 3: Mengubah Teks Menjadi Vektor Angka (*Embedding*) & Menyimpannya ke FAISS

Sekarang kita akan memasukkan seluruh `chunks` di atas ke dalam basis data vektor (**FAISS**):
1. Kita memanggil model **`paraphrase-multilingual-MiniLM-L12-v2`** karena model ini sudah dilatih memahami **Bahasa Indonesia**.
2. Model tersebut mengubah setiap kalimat aturan menjadi deretan angka (**vektor**). Kalimat yang maknanya mirip akan menghasilkan deretan angka yang berdekatan.
3. Seluruh vektor disimpan ke dalam indeks **FAISS** agar bisa dicari dalam hitungan milidetik.

In [3]:
# ==========================================
# LANGKAH 3: EMBEDDING & VECTOR DATABASE
# ==========================================
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Memuat model pengubah teks ke vektor (mendukung Bahasa Indonesia)
print("Sedang memuat model embedding Bahasa Indonesia...")
model_embedding = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 2. Mengubah semua potongan aturan menjadi vektor angka
vektor_dokumen = model_embedding.encode(chunks, normalize_embeddings=True)
vektor_dokumen = np.array(vektor_dokumen, dtype="float32")

# 3. Menyimpan vektor ke dalam database pencarian FAISS
ukuran_vektor = vektor_dokumen.shape[1]
database_vektor = faiss.IndexFlatIP(ukuran_vektor)
database_vektor.add(vektor_dokumen)

print("Selesai! Seluruh aturan kampus sudah tersimpan di Vector Database.")
print(f"Jumlah aturan tersimpan : {database_vektor.ntotal} aturan")

Sedang memuat model embedding Bahasa Indonesia...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Selesai! Seluruh aturan kampus sudah tersimpan di Vector Database.
Jumlah aturan tersimpan : 9 aturan


---
### Langkah 4: Membuat & Menguji Mesin Pencari (*Retriever*)

Sebelum melibatkan *Foundation Model*, mari kita buat fungsi `cari_aturan_relevan()` dan mengujinya terlebih dahulu.

Perhatikan kehebatan pencarian berbasis makna (**Semantic Search**):
Pada contoh uji di bawah, kita bertanya *"Kalau mau **libur kuliah sementara** syaratnya apa?"* — padahal di dalam dokumen tidak ada kata *"libur kuliah"*, melainkan istilah ***"cuti akademik"***. Mari kita lihat apakah mesin pencari kita tetap berhasil menemukan pasal yang tepat!

In [4]:
# ==========================================
# LANGKAH 4: FUNGSI PENCARI (RETRIEVER)
# ==========================================
def cari_aturan_relevan(pertanyaan, jumlah_hasil=2, batas_kemiripan=0.35):
    # Ubah pertanyaan mahasiswa menjadi vektor angka
    vektor_tanya = model_embedding.encode([pertanyaan], normalize_embeddings=True)
    vektor_tanya = np.array(vektor_tanya, dtype="float32")

    # Cari aturan yang paling mirip di database FAISS
    skor, indeks = database_vektor.search(vektor_tanya, jumlah_hasil)

    hasil = []
    for s, idx in zip(skor[0], indeks[0]):
        # Hanya ambil aturan jika skor kemiripannya melewati batas minimal
        if idx != -1 and s >= batas_kemiripan:
            hasil.append({
                "teks": chunks[idx],
                "skor": float(s)
            })
    return hasil

# Uji coba pencarian dengan bahasa sehari-hari (parafrase)
tes_pertanyaan = "Kalau mau libur kuliah sementara syaratnya apa?"
hasil_cari = cari_aturan_relevan(tes_pertanyaan)

print(f"Pertanyaan: '{tes_pertanyaan}'\n")
for nomor, item in enumerate(hasil_cari, start=1):
    print(f"Hasil #{nomor} (Tingkat Kemiripan: {item['skor']:.2f}):")
    print(f"   {item['teks']}\n")

Pertanyaan: 'Kalau mau libur kuliah sementara syaratnya apa?'

Hasil #1 (Tingkat Kemiripan: 0.67):
   [BAB III - Syarat Cuti Akademik] Cuti akademik adalah penundaan kuliah sementara atas izin Rektor. Mahasiswa boleh mengajukan cuti setelah menempuh minimal 2 semester aktif berturut-turut.

Hasil #2 (Tingkat Kemiripan: 0.53):
   [BAB III - Durasi Cuti & Masa Studi] Total cuti akademik selama kuliah S1 maksimal 2 semester dan tidak boleh diambil berturut-turut (kecuali sakit berat dengan surat dokter). Masa cuti tidak mengurangi jatah masa studi. Masa studi normal S1 adalah 8 semester dan batas maksimalnya 14 semester (lewat 14 semester dinyatakan Drop Out).



---
### Langkah 5: Menyiapkan *Foundation Model* (Otak Pembuat Jawaban)

Setelah dokumen yang relevan ditemukan oleh *Retriever*, kita membutuhkan **Foundation Model (LLM)** untuk membaca kutipan pasal tersebut dan menyusunnya menjadi jawaban yang sopan dan mudah dipahami mahasiswa.

* **Default (`PAKAI_GEMINI_API = False`):** Memuat model gratis langsung di Google Colab (`Qwen/Qwen2.5-1.5B-Instruct` jika GPU T4 aktif, atau `google/flan-t5-base` jika hanya memakai CPU).
* **Opsional (`PAKAI_GEMINI_API = True`):** Jika Anda memiliki API Key Gemini di menu **Secrets** Google Colab (`GEMINI_API_KEY`), ubah nilainya menjadi `True`.

In [5]:
# ==========================================
# LANGKAH 5: MEMUAT FOUNDATION MODEL
# ==========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

PAKAI_GEMINI_API = False  # Ubah menjadi True jika ingin memakai Gemini API Key

perangkat = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Perangkat yang digunakan: {perangkat.upper()}")

if PAKAI_GEMINI_API:
    from google import genai
    from google.colab import userdata

    kunci_api = userdata.get("GEMINI_API_KEY")
    client_gemini = genai.Client(api_key=kunci_api)
    print("Foundation Model siap: Google Gemini (gemini-2.5-flash)")
else:
    if perangkat == "cuda":
        nama_model = "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"Memuat model Bahasa Indonesia pada GPU: {nama_model} ...")
        tokenizer = AutoTokenizer.from_pretrained(nama_model)
        model_llm = AutoModelForCausalLM.from_pretrained(
            nama_model,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        mesin_ai = pipeline("text-generation", model=model_llm, tokenizer=tokenizer)
    else:
        nama_model = "google/flan-t5-base"
        print(f"GPU tidak aktif. Memuat model ringan di CPU: {nama_model} ...")
        mesin_ai = pipeline("text2text-generation", model=nama_model)

    print("Foundation Model berhasil dimuat dan siap digunakan!")

Perangkat yang digunakan: CUDA
Memuat model Bahasa Indonesia pada GPU: Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Foundation Model berhasil dimuat dan siap digunakan!


---
### Langkah 6: Menggabungkan Pencari & *Foundation Model* Menjadi Chatbot RAG

Pada fungsi `tanya_chatbot()` di bawah ini, seluruh alur kerja RAG digabungkan:
1. **Cari referensi:** Memanggil `cari_aturan_relevan()` untuk mengambil pasal yang cocok.
2. **Cek ketersediaan info:** Jika tidak ada pasal yang cocok, chatbot langsung menjawab jujur bahwa informasi tidak ditemukan (mencegah AI mengarang).
3. **Susun instruksi (*Prompt*):** Memberikan kutipan pasal kepada *Foundation Model* dengan aturan ketat agar hanya menjawab sesuai kutipan dan menyebutkan sumber Bab-nya.

In [6]:
# ==========================================
# LANGKAH 6: FUNGSI UTAMA CHATBOT RAG
# ==========================================
def tanya_chatbot(pertanyaan, jumlah_hasil=2, batas_kemiripan=0.35, tampilkan_sumber=True):
    # 1. Tahap Retrieval: Cari pasal yang relevan
    referensi = cari_aturan_relevan(
        pertanyaan,
        jumlah_hasil=jumlah_hasil,
        batas_kemiripan=batas_kemiripan
    )

    # Jika tidak ada dokumen yang cocok, tolak secara sopan agar AI tidak mengarang
    if not referensi:
        return (
            "Maaf, informasi tersebut tidak ditemukan di dalam Dokumen Panduan Akademik. "
            "Silakan hubungi bagian Administrasi Akademik (BAAK) kampus."
        )

    # Gabungkan teks referensi yang ditemukan
    konteks_dokumen = "\n".join([item["teks"] for item in referensi])

    if tampilkan_sumber:
        print("Sumber Referensi yang Ditemukan:")
        for item in referensi:
            print(f"  - [Kemiripan: {item['skor']:.2f}] {item['teks']}")
        print("-" * 50)

    # 2. Tahap Augmentation: Beri instruksi ketat ke Foundation Model
    aturan_ai = (
        "Anda adalah Asisten Layanan Akademik Kampus yang ramah dan jelas. "
        "Jawab pertanyaan mahasiswa dalam Bahasa Indonesia HANYA berdasarkan REFERENSI DOKUMEN berikut. "
        "Sebutkan nama BAB yang menjadi rujukan. "
        "Jangan mengarang informasi di luar referensi yang diberikan."
    )

    # 3. Tahap Generation: Hasilkan jawaban akhir
    if PAKAI_GEMINI_API:
        prompt = f"{aturan_ai}\n\nREFERENSI DOKUMEN:\n{konteks_dokumen}\n\nPERTANYAAN:\n{pertanyaan}"
        respons = client_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return respons.text.strip()
    else:
        if perangkat == "cuda":
            pesan = [
                {"role": "system", "content": aturan_ai},
                {"role": "user", "content": f"REFERENSI DOKUMEN:\n{konteks_dokumen}\n\nPERTANYAAN:\n{pertanyaan}"}
            ]
            teks_input = tokenizer.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)
            hasil = mesin_ai(teks_input, max_new_tokens=250, do_sample=False, return_full_text=False)
            return hasil[0]["generated_text"].strip()
        else:
            prompt = f"{aturan_ai}\n\nREFERENSI:\n{konteks_dokumen}\n\nPERTANYAAN: {pertanyaan}\nJAWABAN:"
            hasil = mesin_ai(prompt, max_new_tokens=180)
            return hasil[0]["generated_text"].strip()

print("Chatbot RAG Akademik siap dipakai!")

Chatbot RAG Akademik siap dipakai!


---
### Langkah 7: Uji Coba Kemampuan Chatbot RAG (3 Skenario)

Mari kita uji chatbot kita dengan 3 jenis pertanyaan berbeda:
1. **Pertanyaan Aturan Langsung:** Menanyakan batas maksimal SKS jika IPS 2,80.
2. **Pertanyaan Bahasa Santai (Parafrase):** Menanyakan bolehkah libur kuliah 2 semester berturut-turut.
3. **Pertanyaan di Luar Dokumen (*Out-of-Scope*):** Menanyakan biaya parkir motor (untuk membuktikan chatbot **tidak berhalusinasi**).

In [7]:
# ==========================================
# LANGKAH 7: UJI COBA 3 SKENARIO PERTANYAAN
# ==========================================
contoh_pertanyaan = [
    "Jika IPS semester lalu saya 2,80, maksimal berapa SKS yang bisa saya ambil?",
    "Boleh nggak saya libur kuliah (cuti) selama 2 semester berturut-turut?",
    "Berapa tarif parkir motor per hari di kampus?"
]

for i, tanya in enumerate(contoh_pertanyaan, start=1):
    print(f"=== UJI COBA #{i} ===")
    print(f"Mahasiswa: {tanya}\n")
    jawaban = tanya_chatbot(tanya)
    print(f"Chatbot  : {jawaban}")
    print("\n" + "=" * 60 + "\n")

=== UJI COBA #1 ===
Mahasiswa: Jika IPS semester lalu saya 2,80, maksimal berapa SKS yang bisa saya ambil?

Sumber Referensi yang Ditemukan:
  - [Kemiripan: 0.76] [BAB II - Batas Maksimal SKS] Jumlah maksimal SKS yang dapat diambil semester berikutnya ditentukan berdasarkan IPS semester lalu: IPS >= 3,00 maksimal 24 SKS; IPS 2,50 - 2,99 maksimal 20 SKS; IPS 2,00 - 2,49 maksimal 18 SKS; dan IPS < 2,00 maksimal 15 SKS.
  - [Kemiripan: 0.59] [BAB IV - Syarat Mengambil Skripsi] Mahasiswa S1 boleh mengambil Skripsi jika sudah lulus minimal 110 SKS, IPK minimal 2,75, tidak ada nilai E, nilai D maksimal 2 mata kuliah, serta lulus mata kuliah Metodologi Penelitian minimal nilai B.
--------------------------------------------------
Chatbot  : Berdasarkan referensi dokumen yang diberikan:

- IPS >= 3,00 maksimal 24 SKS
- IPS 2,50 - 2,99 maksimal 20 SKS
- IPS 2,00 - 2,49 maksimal 18 SKS
- IPS < 2,00 maksimal 15 SKS

Dengan IPS semester lalu Anda 2,80, maka sks yang bisa Anda ambil adalah 20 SKS.


---
### Langkah 8: Live Demo Interaktif (Untuk Presentasi Tanya-Jawab)

Gunakan formulir di bawah ini saat sesi presentasi di kelas:
1. Ketik pertanyaan apa saja di kotak **`pertanyaan_mahasiswa`** di sebelah kanan.
2. Klik tombol **Play** di sebelah kiri sel untuk melihat bagaimana Chatbot mencari sumber dan menjawabnya secara langsung!

In [8]:
# ==========================================
# LANGKAH 8: LIVE DEMO CHATBOT AKADEMIK
# ==========================================
#@title Demo Interaktif Chatbot Panduan Akademik { display-mode: "form" }
pertanyaan_mahasiswa = "Apa saja syarat SKS dan IPK supaya saya bisa ambil Skripsi?" #@param {type:"string"}
jumlah_referensi = 3 #@param {type:"slider", min:1, max:4, step:1}
tampilkan_sumber_referensi = True #@param {type:"boolean"}

print(f"Mahasiswa: {pertanyaan_mahasiswa}\n")
jawaban_live = tanya_chatbot(
    pertanyaan=pertanyaan_mahasiswa,
    jumlah_hasil=jumlah_referensi,
    tampilkan_sumber=tampilkan_sumber_referensi
)
print(f"Chatbot  : {jawaban_live}")

Mahasiswa: Apa saja syarat SKS dan IPK supaya saya bisa ambil Skripsi?

Sumber Referensi yang Ditemukan:
  - [Kemiripan: 0.64] [BAB IV - Syarat Mengambil Skripsi] Mahasiswa S1 boleh mengambil Skripsi jika sudah lulus minimal 110 SKS, IPK minimal 2,75, tidak ada nilai E, nilai D maksimal 2 mata kuliah, serta lulus mata kuliah Metodologi Penelitian minimal nilai B.
  - [Kemiripan: 0.51] [BAB II - Batas Maksimal SKS] Jumlah maksimal SKS yang dapat diambil semester berikutnya ditentukan berdasarkan IPS semester lalu: IPS >= 3,00 maksimal 24 SKS; IPS 2,50 - 2,99 maksimal 20 SKS; IPS 2,00 - 2,49 maksimal 18 SKS; dan IPS < 2,00 maksimal 15 SKS.
--------------------------------------------------
Chatbot  : Dalam referensi dokumen tersebut, beberapa syarat untuk mengambil skripsi oleh mahasiswa S1 adalah:

1. Lulus minimal 110 SKS.
2. IPK minimal 2,75.
3. Tidak ada nilai E atau D.
4. Nilai D maksimal 2 mata kuliah.
5. Lulus mata kuliah Metodologi Penelitian dengan nilai B atau lebih tinggi.

Ja

---
## Ringkasan Praktikum
1. **Foundation Model (LLM)** sangat mahir merangkai bahasa, tetapi tidak mengetahui data internal kampus dan berisiko mengarang jawaban (*halusinasi*).
2. **RAG (*Retrieval-Augmented Generation*)** menyelesaikan masalah tersebut dengan cara **mencari potongan dokumen yang relevan terlebih dahulu** (*Retrieval*), lalu menyuruh *Foundation Model* menjawab berdasarkan kutipan tersebut (*Generation*).
3. **Keunggulan Utama RAG:** Jika aturan kampus berubah, kita cukup mengedit teks dokumen pada **Langkah 2** dan menjalankan ulang **Langkah 2 & 3** tanpa perlu melatih ulang model AI yang mahal dan lama.